# 05. Сравнение моделей: WGAN-GP vs VAE vs реальные данные

**Метрики:**
- MMD (Maximum Mean Discrepancy) — общая и по классам
- Frechet Distance — общая и по классам

**Визуализация:**
- t-SNE: реальные vs сгенерированные
- UMAP: то же, другой алгоритм
- Per-class scatter — где модель работает лучше/хуже


In [ ]:
# === Colab setup (skip if running locally) ===
import os, sys

IN_COLAB = "COLAB_GPU" in os.environ
if IN_COLAB:
    # Клонируем репо (первый раз) и переходим в него
    if not os.path.exists("/content/gan-reviews"):
        !git clone https://github.com/SultanKhassenov/gan-reviews.git /content/gan-reviews
    %cd /content/gan-reviews
    !pip install -q -r requirements-trainer.txt

# Добавляем корень репо в sys.path, чтобы импорты работали из notebooks/
sys.path.insert(0, os.path.abspath(".."))

# Mount Google Drive для сохранения чекпоинтов (опционально)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/gan-reviews/checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
else:
    CHECKPOINT_DIR = "../checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("CHECKPOINT_DIR =", CHECKPOINT_DIR)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from metrics.mmd import compute_mmd, compute_mmd_per_class
from metrics.frechet import compute_frechet_distance, compute_frechet_per_class

EMB = Path('data/embeddings')
real = np.load(EMB / 'X_cls.npy').astype('float32')
real_y = np.load(EMB / 'labels.npy')
wgan = np.load(EMB / 'X_gen_wgan.npy').astype('float32')
wgan_y = np.load(EMB / 'y_gen_wgan.npy')
vae = np.load(EMB / 'X_gen_vae.npy').astype('float32')
vae_y = np.load(EMB / 'y_gen_vae.npy')

NUM_CLASSES = int(real_y.max() + 1)


## Метрики


In [ ]:
import pandas as pd

results = []
for name, X, y in [('WGAN-GP', wgan, wgan_y), ('VAE', vae, vae_y)]:
    mmd = compute_mmd(real, X)
    fd = compute_frechet_distance(real, X)
    results.append({'model': name, 'MMD': mmd, 'Frechet': fd})

df_metrics = pd.DataFrame(results)
print(df_metrics)
df_metrics.to_csv('results/metrics_overall.csv', index=False)


In [ ]:
# Метрики по классам
rows = []
for name, X, y in [('WGAN-GP', wgan, wgan_y), ('VAE', vae, vae_y)]:
    mmd_per = compute_mmd_per_class(real, X, real_y, y, NUM_CLASSES)
    fd_per = compute_frechet_per_class(real, X, real_y, y, NUM_CLASSES)
    for c in range(NUM_CLASSES):
        rows.append({
            'model': name, 'class': c,
            'MMD': mmd_per[c], 'Frechet': fd_per[c],
        })
df_per = pd.DataFrame(rows)
print(df_per)
df_per.to_csv('results/metrics_per_class.csv', index=False)


## t-SNE визуализация


In [ ]:
from sklearn.manifold import TSNE

# Объединяем для совместной проекции
n_sample = 1500
idx_r = np.random.choice(len(real), min(n_sample, len(real)), replace=False)
idx_w = np.random.choice(len(wgan), min(n_sample, len(wgan)), replace=False)
idx_v = np.random.choice(len(vae), min(n_sample, len(vae)), replace=False)

X_all = np.vstack([real[idx_r], wgan[idx_w], vae[idx_v]])
src = (['real']*len(idx_r) + ['wgan']*len(idx_w) + ['vae']*len(idx_v))

tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca')
Z = tsne.fit_transform(X_all)

fig, ax = plt.subplots(figsize=(9, 7))
for s, c in zip(['real', 'wgan', 'vae'], ['#1f77b4', '#ff7f0e', '#2ca02c']):
    m = [i for i, v in enumerate(src) if v == s]
    ax.scatter(Z[m, 0], Z[m, 1], c=c, label=s, alpha=0.5, s=10)
ax.legend(); ax.set_title('t-SNE: real vs generated')
plt.savefig('results/tsne_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
